# Airline Passenger Experience & Satisfaction Analytics

| | |
|---|---|
| **Student Name** | Bhatt Tvisha |
| **Project Title** | Airline Passenger Experience & Satisfaction Analytics Dashboard |
| **Dataset** | airline_passenger_satisfaction.csv |
| **Records** | 129,880 rows × 24 columns |
| **Tools** | Python, Pandas, NumPy, Matplotlib, Seaborn, Plotly, Streamlit |

**Objective:** Analyse airline passenger satisfaction data to uncover patterns in service quality, travel class, customer type, travel type, flight delays, and passenger demographics — and deliver actionable business recommendations.

---

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
import warnings

warnings.filterwarnings('ignore')

# Plotting defaults
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (12, 5)
pd.set_option('display.max_columns', 30)
pd.set_option('display.float_format', '{:.2f}'.format)

print('All libraries imported successfully.')

---
## 2. Constants & Helper Functions

All analysis logic is defined here so the notebook is fully self-contained.

In [ ]:
# ── Constants ──────────────────────────────────────────────────────────────
CSV_PATH = 'airline_passenger_satisfaction.csv'

SERVICE_COLS = [
    'Departure and Arrival Time Convenience',
    'Ease of Online Booking',
    'Check-in Service',
    'Online Boarding',
    'Gate Location',
    'On-board Service',
    'Seat Comfort',
    'Leg Room Service',
    'Cleanliness',
    'Food and Drink',
    'In-flight Service',
    'In-flight Wifi Service',
    'In-flight Entertainment',
    'Baggage Handling',
]

AGE_BINS   = [0, 18, 30, 45, 60, 100]
AGE_LABELS = ['Under 18', '18-30', '31-45', '46-60', '60+']

DIST_BINS   = [0, 500, 1500, 3000, 5000]
DIST_LABELS = ['Short (<500)', 'Medium (500-1500)', 'Long (1500-3000)', 'Very Long (3000+)']

# ── Data Loading ───────────────────────────────────────────────────────────
def load_raw_data(path=CSV_PATH):
    """Load the raw CSV without any changes."""
    return pd.read_csv(path)

# ── Data Cleaning ──────────────────────────────────────────────────────────
def clean_data(df):
    """
    Clean the dataframe and return (cleaned_df, cleaning_report).
    Steps:
      1. Record original shape
      2. Remove exact duplicates
      3. Fill 393 missing Arrival Delay values with column median
      4. Ensure numeric types
      5. Remove any rows with invalid Age or Flight Distance
    """
    report = {}
    report['original_rows'] = len(df)
    report['original_cols'] = df.shape[1]
    report['duplicates_found'] = int(df.duplicated().sum())

    # Drop duplicates
    df = df.drop_duplicates().reset_index(drop=True)

    # Handle missing Arrival Delay
    report['missing_arrival_delay'] = int(df['Arrival Delay'].isnull().sum())
    arrival_median = df['Arrival Delay'].median()
    df['Arrival Delay'] = pd.to_numeric(df['Arrival Delay'], errors='coerce').fillna(arrival_median)
    report['arrival_delay_fill_value'] = round(float(arrival_median), 2)

    # Ensure numeric types
    for col in SERVICE_COLS + ['Flight Distance', 'Departure Delay', 'Age']:
        df[col] = pd.to_numeric(df[col], errors='coerce')

    # Remove invalid rows
    rows_before = len(df)
    df = df[(df['Age'] >= 7) & (df['Flight Distance'] > 0)]
    report['invalid_rows_removed'] = rows_before - len(df)
    report['final_rows'] = len(df)
    report['final_cols'] = df.shape[1]

    return df.reset_index(drop=True), report

# ── Feature Engineering ────────────────────────────────────────────────────
def engineer_features(df):
    """
    Add 5 derived analytical columns:
      Total Delay, Delay Status, Age Group, Distance Category, Avg Service Rating
    """
    df = df.copy()
    df['Total Delay']      = df['Departure Delay'] + df['Arrival Delay']
    df['Delay Status']     = df['Total Delay'].apply(lambda x: 'Delayed' if x > 0 else 'On Time')
    df['Age Group']        = pd.cut(df['Age'],  bins=AGE_BINS,  labels=AGE_LABELS,  right=True).astype(str)
    df['Distance Category'] = pd.cut(df['Flight Distance'], bins=DIST_BINS, labels=DIST_LABELS, right=True).astype(str)
    df['Avg Service Rating'] = df[SERVICE_COLS].mean(axis=1).round(2)
    return df

# ── KPI Calculations ───────────────────────────────────────────────────────
def calculate_kpis(df):
    """Return a dict of key metrics for the given (optionally filtered) dataframe."""
    if df.empty:
        return {k: 0 for k in ['Total Passengers','Satisfied Passengers',
                                'Satisfaction Rate (%)','Average Age',
                                'Average Flight Distance','Average Total Delay',
                                'Average Service Rating']}
    total     = len(df)
    satisfied = int((df['Satisfaction'] == 'Satisfied').sum())
    return {
        'Total Passengers':        total,
        'Satisfied Passengers':    satisfied,
        'Satisfaction Rate (%)':   round(satisfied / total * 100, 2),
        'Average Age':             round(df['Age'].mean(), 1),
        'Average Flight Distance': round(df['Flight Distance'].mean(), 1),
        'Average Total Delay':     round(df['Total Delay'].mean(), 1),
        'Average Service Rating':  round(df['Avg Service Rating'].mean(), 2),
    }

# ── Service Rating Helpers ─────────────────────────────────────────────────
def service_rating_summary(df):
    """Mean rating per service area, sorted descending."""
    means = df[SERVICE_COLS].mean().round(3)
    return pd.DataFrame({'Service': means.index, 'Average Rating': means.values}).sort_values(
        'Average Rating', ascending=False).reset_index(drop=True)

def service_rating_by_satisfaction(df):
    """Compare mean service ratings: Satisfied vs Neutral or Dissatisfied."""
    grp = df.groupby('Satisfaction')[SERVICE_COLS].mean().round(3).T
    grp.index.name = 'Service'
    grp = grp.reset_index()
    for cat in ['Satisfied', 'Neutral or Dissatisfied']:
        if cat not in grp.columns:
            grp[cat] = np.nan
    grp['Difference'] = (grp['Satisfied'] - grp['Neutral or Dissatisfied']).round(3)
    return grp.sort_values('Difference', ascending=False).reset_index(drop=True)

# ── Satisfaction Helpers ───────────────────────────────────────────────────
def satisfaction_by_column(df, column):
    """Satisfaction rate (%) per category of `column`."""
    grp = df.groupby(column)['Satisfaction'].apply(
        lambda s: pd.Series({
            'Satisfied':            (s == 'Satisfied').sum(),
            'Total':                len(s),
            'Satisfaction Rate (%)': round((s == 'Satisfied').mean() * 100, 2),
        })
    ).unstack().reset_index()
    grp['Satisfied'] = grp['Satisfied'].astype(int)
    grp['Total']     = grp['Total'].astype(int)
    return grp

def satisfaction_distribution(df):
    """Overall satisfaction counts and percentages."""
    counts = df['Satisfaction'].value_counts().reset_index()
    counts.columns = ['Satisfaction', 'Count']
    counts['Percentage'] = (counts['Count'] / counts['Count'].sum() * 100).round(2)
    return counts

# ── Delay Helper ───────────────────────────────────────────────────────────
def delay_summary(df):
    """Summary statistics for delay-related columns."""
    if df.empty:
        return {}
    return {
        'avg_departure_delay': round(df['Departure Delay'].mean(), 2),
        'avg_arrival_delay':   round(df['Arrival Delay'].mean(), 2),
        'avg_total_delay':     round(df['Total Delay'].mean(), 2),
        'pct_on_time':         round((df['Delay Status'] == 'On Time').mean() * 100, 2),
        'pct_delayed':         round((df['Delay Status'] == 'Delayed').mean() * 100, 2),
    }

print('All helper functions defined.')

---
## 3. Load the Dataset

In [ ]:
raw_df = load_raw_data()
print(f'Dataset shape : {raw_df.shape[0]:,} rows × {raw_df.shape[1]} columns')
raw_df.head()

---
## 4. Data Inspection

In [ ]:
# Column names and data types
print('=== Column Names & Data Types ===')
print(raw_df.dtypes)
print(f'\nTotal columns : {raw_df.shape[1]}')

In [ ]:
# Missing values
print('=== Missing Values ===')
missing = raw_df.isnull().sum()
missing_pct = (missing / len(raw_df) * 100).round(3)
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
print(missing_df[missing_df['Missing Count'] > 0])
print(f'\nTotal missing cells : {missing.sum()}')

In [ ]:
# Duplicates and ID uniqueness
print(f'Duplicate rows      : {raw_df.duplicated().sum()}')
print(f'Unique IDs          : {raw_df["ID"].nunique():,}  (all unique = {raw_df["ID"].nunique() == len(raw_df)})')

In [ ]:
# Unique values in categorical columns
for col in ['Gender', 'Customer Type', 'Type of Travel', 'Class', 'Satisfaction']:
    print(f'{col:25s} : {raw_df[col].unique().tolist()}')

In [ ]:
# Descriptive statistics
raw_df.describe()

In [ ]:
# Service rating value ranges
print('=== Service Column Value Ranges ===')
for col in SERVICE_COLS:
    z = (raw_df[col] == 0).sum()
    print(f'{col:45s} min={raw_df[col].min()}  max={raw_df[col].max()}  zeros={z}')

### Data Quality Summary

| Item | Finding |
|---|---|
| Total rows | 129,880 |
| Total columns | 24 |
| Duplicate rows | **0** — none found |
| Missing values | **393** rows in `Arrival Delay` only (0.30%) |
| ID uniqueness | All 129,880 IDs are unique |
| Service rating range | 0–5 (valid); 0 may mean "not applicable" |
| Age range | 7–85 (valid) |
| Flight Distance | 31–4,983 km (valid) |
| Delay range | 0–1,592 min (right-skewed; extreme outliers exist) |

---
## 5. Data Cleaning

In [ ]:
cleaned_df, cleaning_report = clean_data(raw_df)

print('=== Cleaning Report ===')
for k, v in cleaning_report.items():
    print(f'  {k:35s}: {v}')
print(f'\nCleaned dataframe shape: {cleaned_df.shape}')

**Cleaning Actions Performed:**
1. **No duplicates** to remove.
2. **`Arrival Delay`** — 393 missing values filled with column **median (0.0 min)**.
3. All service-rating columns confirmed within valid range **[0, 5]**.
4. **No invalid Age or Flight Distance** values found.
5. **Original CSV is never overwritten.**

---
## 6. Feature Engineering

In [ ]:
df = engineer_features(cleaned_df)

new_cols = ['Total Delay', 'Delay Status', 'Age Group', 'Distance Category', 'Avg Service Rating']
print('New derived columns:')
print(df[new_cols].head(10))
print(f'\nFinal dataframe shape: {df.shape}')

In [ ]:
# Verify derived features
check = (df['Total Delay'] == df['Departure Delay'] + df['Arrival Delay']).all()
print(f'Total Delay formula verified : {check}')

print('\nAge Group distribution:')
print(df['Age Group'].value_counts().reindex(AGE_LABELS))

print('\nDistance Category distribution:')
print(df['Distance Category'].value_counts().reindex(DIST_LABELS))

print('\nDelay Status distribution:')
print(df['Delay Status'].value_counts())

| New Feature | Formula | Purpose |
|---|---|---|
| `Total Delay` | Departure Delay + Arrival Delay | Combined delay metric |
| `Delay Status` | On Time / Delayed | Binary delay classification |
| `Age Group` | Binned: Under 18 / 18-30 / 31-45 / 46-60 / 60+ | Demographic segmentation |
| `Distance Category` | Short / Medium / Long / Very Long | Distance segmentation |
| `Avg Service Rating` | Mean of all 14 service columns | Per-passenger service score |

---
## 7. Key Performance Indicators

In [ ]:
kpis = calculate_kpis(df)
print('=== Key Performance Indicators (Full Dataset) ===')
for k, v in kpis.items():
    print(f'  {k:30s}: {v}')

---
## 8. Exploratory Data Analysis

### 8.1 Overall Satisfaction Distribution

In [ ]:
sat_dist = satisfaction_distribution(df)
print(sat_dist.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
colors = ['#3b82f6', '#f87171']

axes[0].pie(
    sat_dist['Count'], labels=sat_dist['Satisfaction'],
    autopct='%1.1f%%', colors=colors, startangle=90,
    wedgeprops={'edgecolor': 'white', 'linewidth': 1.5}
)
axes[0].set_title('Overall Satisfaction Distribution', fontweight='bold', fontsize=13)

bars = axes[1].bar(sat_dist['Satisfaction'], sat_dist['Count'], color=colors, edgecolor='white')
for bar, pct in zip(bars, sat_dist['Percentage']):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 400,
                 f'{pct:.1f}%', ha='center', fontweight='bold')
axes[1].set_title('Satisfaction Count', fontweight='bold', fontsize=13)
axes[1].set_ylabel('Number of Passengers')
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
plt.tight_layout()
plt.show()

### 8.2 Passenger Demographics

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
colors_pie = ['#3b82f6', '#60a5fa', '#f87171', '#fca5a5']

for ax, col in zip(axes[0], ['Gender', 'Customer Type', 'Type of Travel']):
    vals = df[col].value_counts()
    ax.pie(vals, labels=vals.index, autopct='%1.1f%%',
           colors=colors_pie[:len(vals)], startangle=90,
           wedgeprops={'edgecolor': 'white', 'linewidth': 1.5})
    ax.set_title(col, fontweight='bold', fontsize=12)

# Class bar
cls = df['Class'].value_counts()
axes[1,0].bar(cls.index, cls.values, color='#3b82f6', edgecolor='white')
axes[1,0].set_title('Travel Class', fontweight='bold')
axes[1,0].set_ylabel('Count')
axes[1,0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))

# Age histogram
axes[1,1].hist(df['Age'], bins=40, color='#3b82f6', edgecolor='white', alpha=0.8)
axes[1,1].set_title('Age Distribution', fontweight='bold')
axes[1,1].set_xlabel('Age')
axes[1,1].set_ylabel('Count')

# Age group bar
ag = df['Age Group'].value_counts().reindex(AGE_LABELS)
axes[1,2].bar(ag.index, ag.values, color='#60a5fa', edgecolor='white')
axes[1,2].set_title('Age Group Distribution', fontweight='bold')
axes[1,2].set_ylabel('Count')
axes[1,2].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))

plt.suptitle('Passenger Demographics', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

### 8.3 Satisfaction by Key Segments

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
palette = 'Blues_d'

for ax, col, title in [
    (axes[0], 'Customer Type',  'By Customer Type'),
    (axes[1], 'Type of Travel', 'By Travel Type'),
    (axes[2], 'Class',          'By Travel Class'),
]:
    sat = satisfaction_by_column(df, col)
    bars = ax.bar(sat[col], sat['Satisfaction Rate (%)'],
                  color=['#3b82f6', '#60a5fa', '#93c5fd'][:len(sat)],
                  edgecolor='white')
    for bar, val in zip(bars, sat['Satisfaction Rate (%)']):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                f'{val:.1f}%', ha='center', fontweight='bold', fontsize=11)
    ax.set_title(f'Satisfaction Rate {title}', fontweight='bold', fontsize=12)
    ax.set_ylabel('Satisfaction Rate (%)')
    ax.set_ylim(0, 90)

plt.suptitle('Satisfaction Analysis by Key Segments', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Satisfaction by Age Group and Delay Status
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, col, order, title in [
    (axes[0], 'Age Group',    AGE_LABELS,        'By Age Group'),
    (axes[1], 'Delay Status', ['On Time','Delayed'], 'By Delay Status'),
]:
    sat = satisfaction_by_column(df, col)
    # Reorder if needed
    sat[col] = pd.Categorical(sat[col], categories=order, ordered=True)
    sat = sat.sort_values(col)
    colors = ['#3b82f6' if r >= 40 else '#f87171' for r in sat['Satisfaction Rate (%)']]
    bars = ax.bar(sat[col].astype(str), sat['Satisfaction Rate (%)'], color=colors, edgecolor='white')
    for bar, val in zip(bars, sat['Satisfaction Rate (%)']):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                f'{val:.1f}%', ha='center', fontweight='bold', fontsize=10)
    ax.set_title(f'Satisfaction Rate {title}', fontweight='bold', fontsize=12)
    ax.set_ylabel('Satisfaction Rate (%)')
    ax.set_ylim(0, 75)

plt.tight_layout()
plt.show()

### 8.4 Service Quality Analysis

In [ ]:
svc_summary = service_rating_summary(df)
print('=== Average Service Ratings (all passengers) ===')
print(svc_summary.to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(12, 7))
svc_sorted = svc_summary.sort_values('Average Rating')

# Color gradient: green for high, red for low
norm = plt.Normalize(svc_sorted['Average Rating'].min(), svc_sorted['Average Rating'].max())
cmap = plt.get_cmap('RdYlGn')
bar_colors = [cmap(norm(v)) for v in svc_sorted['Average Rating']]

bars = ax.barh(svc_sorted['Service'], svc_sorted['Average Rating'],
               color=bar_colors, edgecolor='white')
for bar, val in zip(bars, svc_sorted['Average Rating']):
    ax.text(bar.get_width() + 0.02, bar.get_y() + bar.get_height()/2,
            f'{val:.2f}', va='center', fontweight='bold')
ax.set_xlim(0, 5.3)
ax.set_xlabel('Average Rating (0 - 5)', fontsize=11)
ax.set_title('Average Rating per Service Area', fontweight='bold', fontsize=13)
ax.axvline(x=3.0, color='grey', linestyle='--', alpha=0.5, label='Rating = 3.0')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Service ratings: Satisfied vs Neutral/Dissatisfied
svc_diff = service_rating_by_satisfaction(df)
print('=== Service Rating Gap: Satisfied - Neutral/Dissatisfied ===')
print(svc_diff[['Service','Satisfied','Neutral or Dissatisfied','Difference']].to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(12, 7))
svc_diff_sorted = svc_diff.sort_values('Difference', ascending=True)

bar_colors = ['#3b82f6' if x >= 0 else '#f87171' for x in svc_diff_sorted['Difference']]
bars = ax.barh(svc_diff_sorted['Service'], svc_diff_sorted['Difference'],
               color=bar_colors, edgecolor='white')
for bar, val in zip(bars, svc_diff_sorted['Difference']):
    xpos = bar.get_width() + 0.01 if val >= 0 else bar.get_width() - 0.01
    ax.text(xpos, bar.get_y() + bar.get_height()/2,
            f'{val:+.2f}', va='center', fontweight='bold', fontsize=9)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Rating Difference (Satisfied - Neutral/Dissatisfied)')
ax.set_title('Service Rating Gap: Satisfied vs Neutral/Dissatisfied Passengers',
             fontweight='bold', fontsize=13)
plt.tight_layout()
plt.show()

### 8.5 Flight Delay Analysis

In [ ]:
ds = delay_summary(df)
print('=== Delay Summary ===')
for k, v in ds.items():
    print(f'  {k:25s}: {v}')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, col, color, title in [
    (axes[0], 'Departure Delay', '#3b82f6', 'Departure Delay Distribution (<=300 min)'),
    (axes[1], 'Arrival Delay',   '#f87171', 'Arrival Delay Distribution (<=300 min)'),
    (axes[2], 'Total Delay',     '#a78bfa', 'Total Delay Distribution (<=400 min)'),
]:
    cap = 300 if col != 'Total Delay' else 400
    data = df[df[col] <= cap][col]
    ax.hist(data, bins=60, color=color, edgecolor='white', alpha=0.85)
    ax.set_title(title, fontweight='bold', fontsize=11)
    ax.set_xlabel('Minutes')
    ax.set_ylabel('Count')
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))

plt.suptitle('Flight Delay Distributions', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

### 8.6 Flight Distance Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df['Flight Distance'], bins=60, color='#3b82f6', edgecolor='white', alpha=0.85)
axes[0].set_title('Flight Distance Distribution', fontweight='bold')
axes[0].set_xlabel('Flight Distance (km)')
axes[0].set_ylabel('Count')

dist_sat = satisfaction_by_column(df, 'Distance Category')
dist_sat['Distance Category'] = pd.Categorical(dist_sat['Distance Category'],
                                                categories=DIST_LABELS, ordered=True)
dist_sat = dist_sat.sort_values('Distance Category')
axes[1].bar(dist_sat['Distance Category'].astype(str), dist_sat['Satisfaction Rate (%)'],
            color='#3b82f6', edgecolor='white')
axes[1].set_title('Satisfaction Rate by Distance Category', fontweight='bold')
axes[1].set_ylabel('Satisfaction Rate (%)')
axes[1].tick_params(axis='x', rotation=10)

plt.tight_layout()
plt.show()

---
## 9. Business Questions

In [ ]:
# Q1 & Q2 — Overall satisfaction split
sat_dist = satisfaction_distribution(df)
for _, row in sat_dist.iterrows():
    print(f'Q1/Q2 | {row["Satisfaction"]:30s}: {row["Count"]:,}  ({row["Percentage"]:.2f}%)')

In [ ]:
# Q3 — Customer Type
print('Q3 | Satisfaction by Customer Type:')
ct = satisfaction_by_column(df, 'Customer Type')
print(ct[['Customer Type','Satisfied','Total','Satisfaction Rate (%)']].to_string(index=False))

In [ ]:
# Q4 — Type of Travel
print('Q4 | Satisfaction by Type of Travel:')
tt = satisfaction_by_column(df, 'Type of Travel')
print(tt[['Type of Travel','Satisfied','Total','Satisfaction Rate (%)']].to_string(index=False))

In [ ]:
# Q5 — Travel Class
print('Q5 | Satisfaction by Class:')
cl = satisfaction_by_column(df, 'Class')
print(cl[['Class','Satisfied','Total','Satisfaction Rate (%)']].to_string(index=False))

In [ ]:
# Q6 — Delays vs satisfaction
print('Q6 | Satisfaction by Delay Status:')
dl = satisfaction_by_column(df, 'Delay Status')
print(dl[['Delay Status','Satisfied','Total','Satisfaction Rate (%)']].to_string(index=False))
rates = dl.set_index('Delay Status')['Satisfaction Rate (%)']
print(f'\n   Satisfaction drop due to delays: {rates["On Time"] - rates["Delayed"]:.2f} percentage points')

In [ ]:
# Q7 & Q8 — Best and worst service areas
svc = service_rating_summary(df)
print(f'Q7 | Best service area  : {svc.iloc[0]["Service"]}  ({svc.iloc[0]["Average Rating"]}/5)')
print(f'Q8 | Worst service area : {svc.iloc[-1]["Service"]} ({svc.iloc[-1]["Average Rating"]}/5)')

In [ ]:
# Q9 — Largest rating gap service
svc_diff = service_rating_by_satisfaction(df)
top_gap = svc_diff.iloc[0]
print(f'Q9 | Largest rating gap service: {top_gap["Service"]}')
print(f'     Satisfied = {top_gap["Satisfied"]}   |   Dissatisfied = {top_gap["Neutral or Dissatisfied"]}')
print(f'     Gap = {top_gap["Difference"]}')

In [ ]:
# Q10 — Online Boarding by satisfaction
ob = df.groupby('Satisfaction')['Online Boarding'].mean().round(3)
print('Q10 | Online Boarding avg rating by satisfaction:')
for k, v in ob.items():
    print(f'     {k:30s}: {v:.3f}/5')

In [ ]:
# Q11 — Flight distance vs satisfaction
fd = df.groupby('Satisfaction')['Flight Distance'].mean().round(1)
print('Q11 | Average Flight Distance by satisfaction:')
for k, v in fd.items():
    print(f'     {k:30s}: {v:.1f} km')

In [ ]:
# Q12 — Best and worst passenger segments
seg = df.groupby(['Customer Type', 'Type of Travel', 'Class'])['Satisfaction'].apply(
    lambda s: round((s == 'Satisfied').mean() * 100, 2)
).reset_index()
seg.columns = ['Customer Type', 'Type of Travel', 'Class', 'Satisfaction Rate (%)']

best  = seg.loc[seg['Satisfaction Rate (%)'].idxmax()]
worst = seg.loc[seg['Satisfaction Rate (%)'].idxmin()]

print('Q12 | Best passenger segment:')
print(f'     {best["Customer Type"]} + {best["Type of Travel"]} + {best["Class"]} = {best["Satisfaction Rate (%)"]:.2f}%')
print('Q12 | Worst passenger segment:')
print(f'     {worst["Customer Type"]} + {worst["Type of Travel"]} + {worst["Class"]} = {worst["Satisfaction Rate (%)"]:.2f}%')

In [ ]:
# Q13 — Bottom 3 service areas
print('Q13 | Bottom 3 service areas (improvement priorities):')
for i, row in svc.tail(3).iterrows():
    print(f'     {i+1}. {row["Service"]}  ({row["Average Rating"]}/5)')

---
## 10. Key Findings

In [ ]:
kpis = calculate_kpis(df)
svc  = service_rating_summary(df)
svc_diff = service_rating_by_satisfaction(df)
ct_sat = satisfaction_by_column(df, 'Customer Type').set_index('Customer Type')['Satisfaction Rate (%)']
tt_sat = satisfaction_by_column(df, 'Type of Travel').set_index('Type of Travel')['Satisfaction Rate (%)']
cl_sat = satisfaction_by_column(df, 'Class').set_index('Class')['Satisfaction Rate (%)']
dl_sat = satisfaction_by_column(df, 'Delay Status').set_index('Delay Status')['Satisfaction Rate (%)']

findings = [
    f"FINDING 1: Overall satisfaction is {kpis['Satisfaction Rate (%)']:.2f}% — meaning {100-kpis['Satisfaction Rate (%)']:.2f}% of passengers are Neutral or Dissatisfied.",
    f"FINDING 2: Business class passengers have a {cl_sat['Business']:.2f}% satisfaction rate vs Economy at {cl_sat['Economy']:.2f}% — a {cl_sat['Business']-cl_sat['Economy']:.2f} ppt gap.",
    f"FINDING 3: Business travel passengers are {tt_sat['Business']:.2f}% satisfied vs only {tt_sat['Personal']:.2f}% for Personal travel.",
    f"FINDING 4: Returning passengers ({ct_sat['Returning']:.2f}%) are significantly more satisfied than First-time ({ct_sat['First-time']:.2f}%).",
    f"FINDING 5: '{svc_diff.iloc[0]['Service']}' shows the largest rating gap ({svc_diff.iloc[0]['Difference']:.2f} pts) between satisfied and dissatisfied passengers.",
    f"FINDING 6: '{svc.iloc[-1]['Service']}' is the lowest-rated service area at {svc.iloc[-1]['Average Rating']:.3f}/5.",
    f"FINDING 7: Delayed passengers are {dl_sat['On Time']-dl_sat['Delayed']:.2f} ppt less satisfied than on-time passengers ({dl_sat['Delayed']:.2f}% vs {dl_sat['On Time']:.2f}%).",
]

print('=== KEY FINDINGS ===')
for f in findings:
    print(f'\n  {f}')

---
## 11. Business Recommendations

In [ ]:
recommendations = [
    {
        'finding': f"In-flight Wifi Service has the lowest average rating ({svc.iloc[-1]['Average Rating']}/5).",
        'recommendation': "Upgrade in-flight Wi-Fi infrastructure and investigate passenger feedback on connectivity quality."
    },
    {
        'finding': f"Online Boarding shows the largest satisfaction gap ({svc_diff.iloc[0]['Difference']:.2f} pts) between satisfied and dissatisfied passengers.",
        'recommendation': "Prioritise Online Boarding UX improvements — this single factor most differentiates satisfied from dissatisfied passengers."
    },
    {
        'finding': f"Economy class satisfaction is only {cl_sat['Economy']:.2f}% vs Business class at {cl_sat['Business']:.2f}%.",
        'recommendation': "Review Economy class service touchpoints and identify which service categories differ most from Business class."
    },
    {
        'finding': f"Personal travel passengers have only {tt_sat['Personal']:.2f}% satisfaction (89.87% neutral/dissatisfied).",
        'recommendation': "Design a dedicated service improvement programme for personal travel passengers."
    },
    {
        'finding': f"Delayed passengers are {dl_sat['On Time']-dl_sat['Delayed']:.2f} ppt less satisfied than on-time passengers.",
        'recommendation': "Invest in on-time operations and proactive delay communication to passengers."
    },
    {
        'finding': f"First-time passengers have only {ct_sat['First-time']:.2f}% satisfaction vs {ct_sat['Returning']:.2f}% for Returning passengers.",
        'recommendation': "Build a first-flyer onboarding programme covering booking, check-in, and boarding guidance."
    },
]

print('=== BUSINESS RECOMMENDATIONS ===')
for i, rec in enumerate(recommendations, 1):
    print(f'\nRecommendation {i}')
    print(f'  DATA FINDING    : {rec["finding"]}')
    print(f'  RECOMMENDATION  : {rec["recommendation"]}')

---
## 12. Project Summary

| Step | Description | Status |
|---|---|---|
| 1 | Data Loading | ✅ Loaded 129,880 rows × 24 columns |
| 2 | Data Inspection | ✅ Dtypes, missing values, duplicates confirmed |
| 3 | Data Cleaning | ✅ 393 Arrival Delay NaN values imputed with median |
| 4 | Feature Engineering | ✅ 5 new derived features created |
| 5 | EDA | ✅ Demographics, service ratings, delays, distances |
| 6 | Business Questions | ✅ All 13 questions answered with actual data |
| 7 | Key Findings | ✅ 7 findings derived programmatically |
| 8 | Recommendations | ✅ 6 data-driven recommendations produced |

### Submitted Files

| File | Description |
|---|---|
| `Tvisha_Airline_Passenger_Analytics.ipynb` | This notebook — complete analytical workflow |
| `requirements.txt` | Python dependencies |
| `Tvisha_ProjectReport.docx` | Full project documentation |
| `README.md` | Project overview and setup instructions |

---
_Airline Passenger Experience & Satisfaction Analytics - Bhatt Tvisha_